<a href="https://colab.research.google.com/github/ClementPla/DeepFiberQ/blob/main/DeepFiberQ.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Instructions

Create a data folder on the left panel and upload your images in it.

The accepted formats are tif or czi. But czi seems buggy, so tif preferred for now.

## GPUs

If available, we recommend using a GPU (available on the top right, in the dropdown, select "Change runtime type)"


In [ ]:
!pip uninstall -y dnafiber
!pip install -q git+https://github.com/ClementPla/DNAi

Found existing installation: dnafiber 0.1.950
Uninstalling dnafiber-0.1.950:
  Successfully uninstalled dnafiber-0.1.950
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [6]:
# If you are using jpeg or png files, we expect them to already be processed.
# You can simply read them with PIL or OpenCV.
from pathlib import Path
from dnafiber.deployment import run_one_file
from dnafiber.model.utils import (
    get_ensemble_models,
    _get_model,
    get_error_detection_model,
    Models,
)
from dnafiber.postprocess.fiber import Fibers


In [ ]:
# Either create a single model
model = _get_model(Models.UNET_MOBILEONE_S1)

# Or an ensemble of models (uncomment the following lines to use the ensemble)

# ensemble = get_ensemble_models()

# You can pre-move the models to GPU if you have one
# For ensemble:

# for m in ensemble:
#    m.to("cuda")

# For single model:

model.to("cuda")

# You can also create your own list of models if you want to use a custom ensemble.


In [ ]:
error_detection_model = get_error_detection_model()
# Move to GPU if available
error_detection_model.to("cuda")
# If you don't want to use the error detection model, you can set it to None
# error_detection_model = None

In [ ]:
ROOT_IMG = Path("PATH_TO_YOUR_IMAGE_FILE.CZI")  # or .TIFF # or .JPEG, .PNG, .DV ...
pixel_size = 0.13  # Pixel size in micrometers
device = "cuda"  # or "cpu"
# You may want to iterate through your images and process them in a loop.
# For example, if you have multiple images in a directory, you can use glob or os.listdir to get the file paths.

predicted_fibers: Fibers = run_one_file(
    ROOT_IMG,  # You can also pass a np.ndarray HxWxC if you have already read the image with OpenCV or PIL. Note that in this case, the preprocessing steps (normalization, etc.) are not applied, so you need to make sure that the image is in the correct format for the model.
    model=model,  # or model=ensemble if you are using an ensemble
    pixel_size=pixel_size,
    use_tta=False,  # Set to True to use Test Time Augmentation (TTA)
    error_detection_model=error_detection_model,  # Set to None if you don't want to use error detection
    low_end_hardware=False,  # Set to True if you are using low-end hardware and want to disable some optimizations
    device=device,  # "cuda" or "cpu"
)
# predicted_fibers is a Fibers object containing the predicted fibers and their properties.
# It contains a set of utility functions to save them in a convenient format.

# DNAi is only reliable for double or triple segments fibers. Let's filter any others out.
predicted_fibers = predicted_fibers.valid_copy()  # You can also call predicted_fibers.only_double_copy() or predicted_fibers.only_triple_copy() if you want to keep only double or triple segments fibers, respectively.


# By default, the result of the error_detection_model is stored as a property of each fiber (fiber.proba_error)
# We can filter to keep only fibers with a high probability of being correct (for example, proba_error < 0.5)
correct_fibers = predicted_fibers.filter_errors(0.5)

ratios = predicted_fibers.ratios  # Direct access to the ratios of the fibers

# You can also save the results in a convenient format (for example, CSV or JSON). We first export to a pandas DataFrame, which can then be easily saved to CSV or JSON.
df = correct_fibers.to_df(
    pixel_size=pixel_size, img_name=ROOT_IMG
)  # pixel_size is used to convert the lengths from pixels to micrometers in the DataFrame.
df.to_csv("predicted_fibers.csv", index=False)  # Save to CSV

# If you actually want to keep all the properties of the fibers (including the coordinates of the segments), you can save the Fibers object itself using pickle
correct_fibers.to_pickle("correct_fibers.pkl")

# Finally, you can recreate the labelmap. You do need to provide the shape of the original image
height, width = (
    1024,
    1024,
)  # Replace with the actual height and width of your input image
labelmap = correct_fibers.get_labelmap(
    height, width, fiber_width=3
)  # The labelmap is a 2D array where each fiber is represented by a unique integer value (0 for background, 1 for the first fiber, 2 for the second fiber, etc.).

# Fibers contains a lot of other utility functions to compare two sets of fibers if you have ground truth data, to visualize the fibers, etc.

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_token.py:90: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
